<a href="https://colab.research.google.com/github/serialworm/ai-experiments/blob/main/run.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Mount Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# 2. Reset Environment Variables to fix the Matplotlib Backend error
import os
os.environ['MPLBACKEND'] = 'Agg'

# 3. Setup Python 3.10 & Core Alignment
!apt-get update -qq
!apt-get install -y python3.10 python3.10-venv python3.10-dev build-essential
!update-alternatives --install /usr/bin/python3 python3 /usr/bin/python3.10 10
!update-alternatives --set python3 /usr/bin/python3.10
!curl -sS https://bootstrap.pypa.io/get-pip.py | python3.10

# 4. Fix Dependencies (NumPy 1.x, Pydantic 1.x, and CLIP)
!python3.10 -m pip install "numpy<2" "Pillow<11" "pydantic<2" "matplotlib<3.9"
!python3.10 -m pip install ftfy regex tqdm omegaconf pytorch-lightning==1.9.4 torchmetrics==0.11.4
# Using --no-build-isolation allows CLIP to see the 'wheel' and 'setuptools' we installed
!python3.10 -m pip install wheel setuptools
!python3.10 -m pip install git+https://github.com/openai/CLIP.git@d50d76daa670286dd6cacf3bcd80b5e4823fc8e1 --no-build-isolation

# 5. Clone WebUI and Mirror
%cd /content
if not os.path.exists('stable-diffusion-webui'):
    !git clone https://github.com/AUTOMATIC1111/stable-diffusion-webui
%cd /content/stable-diffusion-webui

!mkdir -p repositories
if not os.path.exists('repositories/stable-diffusion-stability-ai'):
    !git clone https://github.com/w-e-w/stablediffusion.git repositories/stable-diffusion-stability-ai

# 6. Hybrid Symlinks
drive_path = "/content/drive/MyDrive/stable-diffusion-webui-colab"
os.makedirs(f"{drive_path}/Stable-diffusion", exist_ok=True)
os.makedirs(f"{drive_path}/Lora", exist_ok=True)

def safe_symlink(source, target):
    if os.path.islink(target): os.unlink(target)
    elif os.path.isdir(target): import shutil; shutil.rmtree(target)
    !ln -s "{source}" "{target}"

safe_symlink(f"{drive_path}/Stable-diffusion", "/content/stable-diffusion-webui/models/Stable-diffusion")
safe_symlink(f"{drive_path}/Lora", "/content/stable-diffusion-webui/models/Lora")

# 7. Launch
# --skip-torch-cuda-test is added just in case the backend error masked a driver check
!python3.10 launch.py --share --xformers --enable-insecure-extension-access --skip-python-version-check --theme dark